In [ ]:
# CELDA 1 — Instalación
!pip -q install pyspark

In [ ]:
# CELDA 2 — Sesión Spark en modo local
import time, statistics, os
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("DataSalud_Peru_INEN")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())

print("Spark", spark.version, "| núcleos Spark:", spark.sparkContext.defaultParallelism, "| CPU Colab:", os.cpu_count())

Spark 4.0.4 | núcleos Spark: 2 | CPU Colab: 2


In [ ]:
# CELDA 4 — Subir los 2 CSV (selecciónalos en el cuadro que aparece)
from google.colab import files
subidos = files.upload()
print(list(subidos.keys()))

Saving nuevos.csv to nuevos.csv
['nuevos.csv']


In [ ]:
from google.colab import files
subidos = files.upload()
print(list(subidos.keys()))

Saving egresos.csv to egresos.csv
['egresos.csv']


In [ ]:
import os
print([f for f in os.listdir(".") if f.endswith(".csv")])

['egresos.csv', 'nuevos.csv']


In [ ]:
RUTA_NUEVOS  = "nuevos.csv"
RUTA_EGRESOS = "egresos.csv"

opts = dict(header=True, inferSchema=True, sep=",", encoding="UTF-8")

# Deja los nombres de columna en MAYÚSCULAS y sin espacios ni BOM, así el código no depende de cómo vengan escritos
def normalizar(df):
    return df.toDF(*[c.replace("\ufeff", "").strip().upper() for c in df.columns])

t0 = time.perf_counter()
nuevos  = normalizar(spark.read.csv(RUTA_NUEVOS,  **opts))
egresos = normalizar(spark.read.csv(RUTA_EGRESOS, **opts))
filas_n, filas_e = nuevos.count(), egresos.count()
t_lectura = (time.perf_counter() - t0) * 1000

print(f"Pacientes nuevos: {filas_n} | Egresos: {filas_e} | Lectura de ambos CSV: {t_lectura:.0f} ms")
print("Columnas nuevos :", nuevos.columns)
print("Columnas egresos:", egresos.columns)

Pacientes nuevos: 78974 | Egresos: 54635 | Lectura de ambos CSV: 11315 ms
Columnas nuevos : ['2BDE508A64EAF3D04DAB77A4354DD86E', 'CE4846DF7606A867', '2024-04-17', 'FEMENINO', '63', '150143', 'LIMA-LIMA-VILLA MARIA DEL TRIUNFO', '3', '2026-08-17']
Columnas egresos: ['C84A158B0F5A917784A23F6C70EB731D', 'B5D8BB640D857739', 'FEMENINO', '95', '2023-01-05', '2023-01-12', '150801', 'LIMA-HUAURA-HUACHO', '2026-08-17']


In [ ]:
# Definimos los nombres correctos para las columnas de cada archivo, ya que no traen cabecera
columnas_nuevos = ["ID_PACIENTE", "ID_ESTABLECIMIENTO", "FECHA_INGRESO", "SEXO", "EDAD", "UBIGEO", "DEPARTAMENTO_PROVINCIA_DISTRITO", "TIPO_INGRESO", "FECHA_CORTE"]
columnas_egresos = ["ID_REGISTRO", "ID_PACIENTE", "SEXO", "EDAD", "FECHA_INGRESO", "FECHA_EGRESO", "UBIGEO", "DEPARTAMENTO_PROVINCIA_DISTRITO", "FECHA_CORTE"]

# Leemos los CSV como STRING puro (inferSchema=False) para evitar fallos de parser automáticos
opts_seguros = dict(header=False, inferSchema=False, sep=",", encoding="UTF-8")

nuevos_crudo = spark.read.csv(RUTA_NUEVOS, **opts_seguros)
egresos_crudo = spark.read.csv(RUTA_EGRESOS, **opts_seguros)

# Asignamos los nombres de columna definidos
nuevos = nuevos_crudo.toDF(*columnas_nuevos)
egresos = egresos_crudo.toDF(*columnas_egresos)

# Evita errores si una fecha no coincide con el formato probado
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

# Acepta fechas como 20250620 o como 2025-06-20 de forma segura
def a_fecha(c):
    s = F.col(c).cast("string")
    return F.coalesce(F.to_date(s, "yyyy-MM-dd"), F.to_date(s, "yyyyMMdd"))

egresos = (egresos
           .withColumn("FECHA_INGRESO", a_fecha("FECHA_INGRESO"))
           .withColumn("FECHA_EGRESO",  a_fecha("FECHA_EGRESO"))
           .withColumn("EDAD", F.col("EDAD").cast("int"))
           .withColumn("UBIGEO", F.col("UBIGEO").cast("int")))

nuevos = (nuevos
          .withColumn("FECHA_INGRESO", a_fecha("FECHA_INGRESO"))
          .withColumn("EDAD", F.col("EDAD").cast("int"))
          .withColumn("UBIGEO", F.col("UBIGEO").cast("int")))

nuevos.cache().count(); egresos.cache().count()
nuevos.createOrReplaceTempView("pacientes_nuevos")
egresos.createOrReplaceTempView("egresos")

# Validación: las fechas nulas deben ser pocas (si son casi todas, el formato no coincide)
egresos.select(
    F.count("*").alias("filas"),
    F.sum(F.col("FECHA_INGRESO").isNull().cast("int")).alias("ingreso_nulo"),
    F.sum(F.col("FECHA_EGRESO").isNull().cast("int")).alias("egreso_nulo")).show()

+-----+------------+-----------+
|filas|ingreso_nulo|egreso_nulo|
+-----+------------+-----------+
|54636|           0|          0|
+-----+------------+-----------+



In [ ]:
def q1_sobre(df):
    return (df.groupBy("SEXO")
            .agg(F.count("*").alias("pacientes"),
                 F.round(F.avg("EDAD"), 2).alias("edad_prom"),
                 F.min("EDAD").alias("edad_min"),
                 F.max("EDAD").alias("edad_max"),
                 F.round(F.stddev("EDAD"), 2).alias("edad_desv")))

def q1():
    return q1_sobre(nuevos)

q1().show()

+---------+---------+---------+--------+--------+---------+
|     SEXO|pacientes|edad_prom|edad_min|edad_max|edad_desv|
+---------+---------+---------+--------+--------+---------+
| FEMENINO|    50469|    52.03|       0|     111|    17.93|
|MASCULINO|    28506|    55.02|       0|     103|    21.52|
+---------+---------+---------+--------+--------+---------+



In [ ]:
SQL_Q2 = """
SELECT RangoEdad, COUNT(*) AS egresos,
       ROUND(AVG(DATEDIFF(FECHA_EGRESO, FECHA_INGRESO)), 2) AS estancia_prom
FROM (
    SELECT CASE WHEN EDAD < 18 THEN 'Menor'
                WHEN EDAD BETWEEN 18 AND 25 THEN 'Joven'
                WHEN EDAD BETWEEN 26 AND 59 THEN 'Adulto'
                ELSE 'Adulto Mayor' END AS RangoEdad,
           FECHA_INGRESO, FECHA_EGRESO
    FROM egresos
    WHERE FECHA_INGRESO IS NOT NULL AND FECHA_EGRESO IS NOT NULL
      AND FECHA_INGRESO <= FECHA_EGRESO) x
GROUP BY RangoEdad
"""

def q2():
    return spark.sql(SQL_Q2)

q2().show()

+------------+-------+-------------+
|   RangoEdad|egresos|estancia_prom|
+------------+-------+-------------+
|      Adulto|  25079|         6.57|
|       Joven|   3295|        13.19|
|Adulto Mayor|  21854|         4.86|
|       Menor|   4408|         27.1|
+------------+-------+-------------+



In [ ]:
def medir(fn, corridas=5):
    t = []
    for _ in range(corridas):
        t0 = time.perf_counter()
        fn().collect()
        t.append((time.perf_counter() - t0) * 1000)
    return t

resultados_spark = {}
for nombre, fn in [("Q1", q1), ("Q2", q2)]:
    t = medir(fn)
    resultados_spark[nombre] = statistics.mean(t[1:])
    print(f"{nombre}: 1ª corrida = {t[0]:.0f} ms | promedio corridas 2-5 = {resultados_spark[nombre]:.0f} ms")

Q1: 1ª corrida = 321 ms | promedio corridas 2-5 = 400 ms
Q2: 1ª corrida = 290 ms | promedio corridas 2-5 = 279 ms
